# PhasePhyto Apple Overlap Colab Pipeline

This notebook downloads the three apple-overlap source datasets into Google Drive, stores them as **tar archives** to save memory and speed up future sessions, prepares the strict shared-label overlap subset, archives that overlap subset too, then hydrates the overlap archive onto Colab SSD for training/evaluation.

> **If you edit any cell, click `Runtime -> Restart runtime` and `Run all`.**
> All real logic lives in `scripts/prepare_overlap_datasets.py`,
> `scripts/download_data.py`, and `configs/apple_overlap_*.yaml`. The notebook
> is a thin driver around those — change the script or the config, push the
> branch in `CONFIG['repo_branch']`, then re-run.

Shared labels:

- `Apple___healthy`
- `Apple___Apple_scab`
- `Apple___Cedar_apple_rust`

Datasets used:

- PlantVillage
- PlantDoc
- Plant Pathology 2021


In [ ]:
# ============================================================
# CONFIGURATION -- edit only if you want different paths/behavior
# ============================================================

CONFIG = {
    "drive_project_dir": "/content/drive/MyDrive/PhasePhyto",
    "repo_url": "https://github.com/kautilyaa/PhasePhyto.git",
    "repo_branch": "new_updation",
    "repo_dir": "/content/PhasePhyto",
    "kaggle_json_drive_path": "/content/drive/MyDrive/kaggle.json",
    "download_plantvillage": True,
    "download_plantdoc": True,
    "download_plant_pathology_2021": True,
    "force_redownload": False,
    "recreate_archives": False,
    "rebuild_overlap": False,
    "allow_partial_overlap": False,
    "overlap_mode": "copy",  # use copy so the overlap archive is self-contained
    "hydrate_overlap_to_ssd": True,
    "keep_local_stage_data": False,
    "run_train": False,
    "run_eval_plantdoc": False,
    "run_eval_pp2021": False,
    "checkpoint_dir": "/content/drive/MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc",
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")


In [2]:
# ============================================================
# MOUNT DRIVE + CLONE/INSTALL REPO + DEFINE PATHS
# ============================================================

from pathlib import Path
import os
import shutil
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path(CONFIG["repo_dir"])
if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", CONFIG["repo_branch"],
        CONFIG["repo_url"], str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", CONFIG["repo_branch"], "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--quiet"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DRIVE_PROJECT_DIR = Path(CONFIG["drive_project_dir"])
ARCHIVE_DIR = DRIVE_PROJECT_DIR / "data" / "archives"
OVERLAP_ARCHIVE = ARCHIVE_DIR / "apple_strict.tar"
LOCAL_STAGE_ROOT = Path("/content/phasephyto_stage")
LOCAL_RAW_PARENT = LOCAL_STAGE_ROOT / "raw"
LOCAL_BENCHMARK_PARENT = LOCAL_STAGE_ROOT / "benchmarks"
LOCAL_OVERLAP_BUILD_PARENT = LOCAL_STAGE_ROOT / "overlap"
LOCAL_OVERLAP_BUILD_ROOT = LOCAL_OVERLAP_BUILD_PARENT / "apple_strict"
LOCAL_DATA_ROOT = Path("/content/data")
LOCAL_OVERLAP_PARENT = LOCAL_DATA_ROOT / "overlap"
LOCAL_OVERLAP_ROOT = LOCAL_OVERLAP_PARENT / "apple_strict"

PLANTVILLAGE_ARCHIVE = ARCHIVE_DIR / "plantvillage.tar"
PLANTDOC_ARCHIVE = ARCHIVE_DIR / "plantdoc.tar"
PP2021_ARCHIVE = ARCHIVE_DIR / "plant_pathology_2021.tar"

LOCAL_PLANT_DISEASE_ROOT = LOCAL_RAW_PARENT / "plant_disease"
LOCAL_BENCHMARK_ROOT = LOCAL_BENCHMARK_PARENT / "plant_benchmarks"
PLANTVILLAGE_DIR = LOCAL_PLANT_DISEASE_ROOT / "plantvillage"
PLANTDOC_DIR = LOCAL_PLANT_DISEASE_ROOT / "plantdoc"
PP2021_DIR = LOCAL_BENCHMARK_ROOT / "plant_pathology_2021"

for path in [
    DRIVE_PROJECT_DIR,
    ARCHIVE_DIR,
    LOCAL_STAGE_ROOT,
    LOCAL_RAW_PARENT,
    LOCAL_BENCHMARK_PARENT,
    LOCAL_OVERLAP_BUILD_PARENT,
    LOCAL_DATA_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Repo ready:", REPO_DIR)
print("Archive dir:", ARCHIVE_DIR)
print("Local raw parent:", LOCAL_RAW_PARENT)
print("Local benchmark parent:", LOCAL_BENCHMARK_PARENT)
print("Local overlap build root:", LOCAL_OVERLAP_BUILD_ROOT)
print("Local hydrated overlap root:", LOCAL_OVERLAP_ROOT)


Mounted at /content/drive
Repo ready: /content/PhasePhyto
Archive dir: /content/drive/MyDrive/PhasePhyto/data/archives
Local raw parent: /content/phasephyto_stage/raw
Local benchmark parent: /content/phasephyto_stage/benchmarks
Local overlap build root: /content/phasephyto_stage/overlap/apple_strict
Local hydrated overlap root: /content/data/overlap/apple_strict


In [ ]:
# ============================================================
# HELPERS + KAGGLE CREDENTIALS
# ============================================================
# This notebook is a driver. The canonical logic lives in the repo:
#   - scripts/download_data.py             (raw dataset downloads)
#   - scripts/prepare_overlap_datasets.py  (strict apple overlap inspect/build)
#   - configs/apple_overlap_*.yaml         (committed training configs)
# Do NOT duplicate that logic here. To change behavior, edit the script or
# config in the repo and push the branch named in CONFIG['repo_branch'].

import json
import shutil
import sys
import tarfile
from pathlib import Path

# scripts/ is not an installed package, so put it on sys.path explicitly.
SCRIPTS_DIR = REPO_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import download_data  # noqa: E402
from prepare_overlap_datasets import (  # noqa: E402
    APPLE_STRICT_CLASSES,
    inspect_apple_overlap,
    prepare_apple_overlap,
)

CONFIGS_DIR = REPO_DIR / "configs"
PLANTDOC_CFG = CONFIGS_DIR / "apple_overlap_plantdoc.yaml"
PP2021_CFG = CONFIGS_DIR / "apple_overlap_pp2021.yaml"
for cfg in (PLANTDOC_CFG, PP2021_CFG):
    if not cfg.exists():
        raise FileNotFoundError(
            f"Missing committed config: {cfg}. Make sure CONFIG['repo_branch'] "
            f"({CONFIG['repo_branch']}) contains the apple-overlap configs."
        )


def run(cmd, *, capture: bool = False):
    print("\n$ " + " ".join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        check=False,
        text=True,
        capture_output=capture,
    )
    if capture and result.stdout:
        print(result.stdout)
    if capture and result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {result.returncode}): " + " ".join(map(str, cmd))
        )
    return result


def ensure_kaggle_credentials() -> None:
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    drive_copy = Path(CONFIG["kaggle_json_drive_path"])

    if kaggle_json.exists():
        os.chmod(kaggle_json, 0o600)
        print("Kaggle credentials already exist:", kaggle_json)
        return

    if drive_copy.exists():
        shutil.copy2(drive_copy, kaggle_json)
        os.chmod(kaggle_json, 0o600)
        print("Copied Kaggle credentials from Drive:", drive_copy)
        return

    print("Upload kaggle.json now (Kaggle > Account > Create New API Token).")
    from google.colab import files
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise RuntimeError("kaggle.json was not uploaded.")
    shutil.move("kaggle.json", kaggle_json)
    shutil.copy2(kaggle_json, drive_copy)
    os.chmod(kaggle_json, 0o600)
    print("Saved reusable kaggle.json to:", drive_copy)


def create_or_refresh_tar(source_dir: Path, archive_path: Path, *, force: bool) -> None:
    if not source_dir.exists():
        raise FileNotFoundError(f"Source directory missing: {source_dir}")
    if archive_path.exists() and not force:
        print(f"Archive already exists: {archive_path}")
        return
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    if archive_path.exists():
        archive_path.unlink()
    with tarfile.open(archive_path, "w") as tf:
        tf.add(source_dir, arcname=source_dir.name)
    size_gb = archive_path.stat().st_size / 1e9
    print(f"Created archive: {archive_path} ({size_gb:.2f} GB)")


def extract_tar_to_parent(archive_path: Path, parent_dir: Path, *, clean_target: bool = True) -> Path:
    if not archive_path.exists():
        raise FileNotFoundError(f"Archive not found: {archive_path}")
    with tarfile.open(archive_path) as tf:
        top_levels = [m.name.split("/", 1)[0] for m in tf.getmembers() if m.name]
        top_level = next((n for n in top_levels if n and n != "."), None)
        if top_level is None:
            raise RuntimeError(f"Could not determine top-level folder inside {archive_path}")
    target_dir = parent_dir / top_level
    if clean_target and target_dir.exists():
        shutil.rmtree(target_dir)
    parent_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path) as tf:
        tf.extractall(parent_dir)
    print(f"Extracted {archive_path} -> {target_dir}")
    return target_dir


ensure_kaggle_credentials()


## Preflight: what will be reused vs rebuilt

This cell checks which tar archives already exist on Drive and tells you whether the notebook will reuse them or rebuild/download them based on the current config flags.


In [4]:
# ============================================================
# PREFLIGHT: DRIVE TAR STATUS + PLANNED ACTIONS
# ============================================================

preflight_rows = [
    ("plantvillage", PLANTVILLAGE_ARCHIVE, CONFIG["download_plantvillage"]),
    ("plantdoc", PLANTDOC_ARCHIVE, CONFIG["download_plantdoc"]),
    ("plant_pathology_2021", PP2021_ARCHIVE, CONFIG["download_plant_pathology_2021"]),
    ("apple_overlap", OVERLAP_ARCHIVE, True),
]

print("Preflight status:\n")
for name, archive_path, enabled in preflight_rows:
    exists = archive_path.exists()
    status = "present" if exists else "missing"

    if name == "apple_overlap":
        if exists and not CONFIG["rebuild_overlap"] and not CONFIG["recreate_archives"]:
            action = "reuse overlap tar from Drive"
        elif CONFIG["rebuild_overlap"] or CONFIG["recreate_archives"]:
            action = "rebuild overlap + refresh tar"
        else:
            action = "build overlap and save tar"
    else:
        if not enabled:
            action = "skip (disabled in CONFIG)"
        elif exists and not CONFIG["force_redownload"] and not CONFIG["recreate_archives"]:
            action = "reuse tar from Drive and unpack to SSD"
        else:
            action = "download/prep on SSD and save tar to Drive"

    print(f"- {name}: {status} | {archive_path}")
    print(f"  planned action: {action}")

print("\nFlag summary:")
print(f"  force_redownload={CONFIG['force_redownload']}")
print(f"  recreate_archives={CONFIG['recreate_archives']}")
print(f"  rebuild_overlap={CONFIG['rebuild_overlap']}")
print(f"  allow_partial_overlap={CONFIG['allow_partial_overlap']}")
print(f"  hydrate_overlap_to_ssd={CONFIG['hydrate_overlap_to_ssd']}")


Preflight status:

- plantvillage: present | /content/drive/MyDrive/PhasePhyto/data/archives/plantvillage.tar
  planned action: reuse tar from Drive and unpack to SSD
- plantdoc: present | /content/drive/MyDrive/PhasePhyto/data/archives/plantdoc.tar
  planned action: reuse tar from Drive and unpack to SSD
- plant_pathology_2021: present | /content/drive/MyDrive/PhasePhyto/data/archives/plant_pathology_2021.tar
  planned action: reuse tar from Drive and unpack to SSD
- apple_overlap: missing | /content/drive/MyDrive/PhasePhyto/data/archives/apple_strict.tar
  planned action: build overlap and save tar

Flag summary:
  force_redownload=False
  recreate_archives=False
  rebuild_overlap=False
  allow_partial_overlap=False
  hydrate_overlap_to_ssd=True


In [ ]:
# ============================================================
# DOWNLOAD RAW DATASETS TO LOCAL SSD, THEN SAVE TAR ARCHIVES TO DRIVE
# ============================================================
# Downloads are delegated to scripts.download_data so this notebook stays in
# lockstep with the canonical dataset prep code.

dataset_jobs = [
    {
        "enabled": CONFIG["download_plantvillage"],
        "name": "plantvillage",
        "archive": PLANTVILLAGE_ARCHIVE,
        "expected_dir": PLANTVILLAGE_DIR,
        "download_fn": download_data.download_plantvillage,
    },
    {
        "enabled": CONFIG["download_plantdoc"],
        "name": "plantdoc",
        "archive": PLANTDOC_ARCHIVE,
        "expected_dir": PLANTDOC_DIR,
        "download_fn": download_data.download_plantdoc,
    },
    {
        "enabled": CONFIG["download_plant_pathology_2021"],
        "name": "plant_pathology_2021",
        "archive": PP2021_ARCHIVE,
        "expected_dir": PP2021_DIR,
        "download_fn": download_data.download_plant_pathology_2021,
    },
]

for job in dataset_jobs:
    if not job["enabled"]:
        print(f"Skipping {job['name']} (disabled in CONFIG).")
        continue

    archive_path = job["archive"]
    expected_dir = job["expected_dir"]

    if archive_path.exists() and not CONFIG["force_redownload"] and not CONFIG["recreate_archives"]:
        print(f"Using existing archive for {job['name']}: {archive_path}")
        extract_tar_to_parent(archive_path, expected_dir.parent, clean_target=True)
        continue

    if expected_dir.exists():
        shutil.rmtree(expected_dir)
    expected_dir.parent.mkdir(parents=True, exist_ok=True)
    job["download_fn"](expected_dir.parent)
    create_or_refresh_tar(expected_dir, archive_path, force=True)


In [6]:
# ============================================================
# ARCHIVE STATUS ON DRIVE
# ============================================================

print("Raw archive status:")
for archive_path in [PLANTVILLAGE_ARCHIVE, PLANTDOC_ARCHIVE, PP2021_ARCHIVE]:
    status = "present" if archive_path.exists() else "missing"
    print(f"  {archive_path.name}: {status}")
overlap_status = "present" if OVERLAP_ARCHIVE.exists() else "missing"
print(f"Overlap archive status: {OVERLAP_ARCHIVE.name}: {overlap_status}")
print(
    "If a later session loses local SSD data, rerun the archive cells to "
    "unpack existing Drive tar files instead of re-downloading."
)


Raw archive status:
  plantvillage.tar: present
  plantdoc.tar: present
  plant_pathology_2021.tar: present
Overlap archive status: apple_strict.tar: missing
If a later session loses local SSD data, rerun the archive cells to unpack existing Drive tar files instead of re-downloading.


## Overlap coverage precheck

This cell inspects the apple-overlap coverage across PlantVillage, PlantDoc, and Plant Pathology 2021 before attempting to build the overlap subset. If a class is missing, you can either fix the source dataset or set `allow_partial_overlap=True` to continue with a partial subset.


In [ ]:
# ============================================================
# PREFLIGHT: APPLE OVERLAP COVERAGE REPORT
# ============================================================

overlap_report = inspect_apple_overlap(
    plantvillage_root=PLANTVILLAGE_DIR,
    plantdoc_root=PLANTDOC_DIR,
    plant_pathology_2021_root=PP2021_DIR,
)
print(json.dumps(overlap_report, indent=2))
if not CONFIG["allow_partial_overlap"] and not overlap_report["is_complete"]:
    raise RuntimeError(
        "Strict apple overlap is incomplete. Set CONFIG['allow_partial_overlap']=True "
        "to continue with a partial subset, or fix the missing classes shown above."
    )


In [ ]:
# ============================================================
# BUILD STRICT APPLE OVERLAP ONLY IF NEEDED, THEN SAVE TAR TO DRIVE
# ============================================================

if OVERLAP_ARCHIVE.exists() and not CONFIG["rebuild_overlap"] and not CONFIG["recreate_archives"]:
    print(f"Using existing overlap archive from Drive: {OVERLAP_ARCHIVE}")
else:
    if CONFIG["rebuild_overlap"] and LOCAL_OVERLAP_BUILD_ROOT.exists():
        shutil.rmtree(LOCAL_OVERLAP_BUILD_ROOT)

    overlap_report = prepare_apple_overlap(
        plantvillage_root=PLANTVILLAGE_DIR,
        plantdoc_root=PLANTDOC_DIR,
        plant_pathology_2021_root=PP2021_DIR,
        output_root=LOCAL_OVERLAP_BUILD_ROOT,
        mode=CONFIG["overlap_mode"],
        require_all_classes=not CONFIG["allow_partial_overlap"],
        clean=True,
    )
    print(json.dumps(overlap_report, indent=2))
    create_or_refresh_tar(
        LOCAL_OVERLAP_BUILD_ROOT,
        OVERLAP_ARCHIVE,
        force=CONFIG["recreate_archives"] or CONFIG["rebuild_overlap"],
    )


In [ ]:
# ============================================================
# HYDRATE OVERLAP TAR TO COLAB SSD (/content/data) FOR TRAINING SPEED
# ============================================================

if CONFIG["hydrate_overlap_to_ssd"]:
    if not OVERLAP_ARCHIVE.exists():
        raise FileNotFoundError(
            f"Cannot hydrate: overlap archive missing at {OVERLAP_ARCHIVE}. "
            "This usually means cell 10 was skipped (e.g. allow_partial_overlap=True "
            "with no prior tar). Set CONFIG['rebuild_overlap']=True or fix overlap "
            "coverage and rerun cell 10 first."
        )
    if LOCAL_OVERLAP_PARENT.exists():
        shutil.rmtree(LOCAL_OVERLAP_PARENT)
    extract_tar_to_parent(OVERLAP_ARCHIVE, LOCAL_OVERLAP_PARENT, clean_target=False)
    print("Hydrated overlap archive to SSD:", LOCAL_OVERLAP_ROOT)
else:
    print("Skipped overlap hydration; use the overlap archive directly later if you want.")

if not CONFIG["keep_local_stage_data"]:
    if LOCAL_RAW_PARENT.exists():
        shutil.rmtree(LOCAL_RAW_PARENT)
    if LOCAL_BENCHMARK_PARENT.exists():
        shutil.rmtree(LOCAL_BENCHMARK_PARENT)
    if LOCAL_OVERLAP_BUILD_ROOT.exists():
        shutil.rmtree(LOCAL_OVERLAP_BUILD_ROOT)
    print("Cleaned local stage folders after archiving.")
else:
    print("Keeping local stage folders on SSD because CONFIG['keep_local_stage_data'] = True.")


In [10]:
# ============================================================
# SANITY CHECK COUNTS
# ============================================================

for dataset_name in ["plantvillage", "plantdoc", "plant_pathology_2021"]:
    ds_root = LOCAL_OVERLAP_ROOT / dataset_name
    print(f"\nDATASET: {dataset_name}")
    for class_name in APPLE_STRICT_CLASSES:
        class_dir = ds_root / class_name
        n = len([p for p in class_dir.glob("*") if p.is_file()]) if class_dir.exists() else 0
        print(f"  {class_name}: {n}")



DATASET: plantvillage
  Apple___healthy: 1645
  Apple___Apple_scab: 630
  Apple___Cedar_apple_rust: 275

DATASET: plantdoc
  Apple___healthy: 9
  Apple___Apple_scab: 10
  Apple___Cedar_apple_rust: 10

DATASET: plant_pathology_2021
  Apple___healthy: 4624
  Apple___Apple_scab: 4826
  Apple___Cedar_apple_rust: 1860


In [ ]:
# ============================================================
# TRAIN: PlantVillage source -> PlantDoc target overlap benchmark
# ============================================================

if CONFIG["run_train"]:
    run([
        sys.executable, "-m", "phasephyto.train",
        "--config", str(PLANTDOC_CFG),
        "--override",
        f"checkpoint_dir={CONFIG['checkpoint_dir']}",
        f"data.root={LOCAL_OVERLAP_ROOT}",
        f"data.source_dir={LOCAL_OVERLAP_ROOT / 'plantvillage'}",
        f"data.target_dir={LOCAL_OVERLAP_ROOT / 'plantdoc'}",
        f"data.eval_source_dir={LOCAL_OVERLAP_ROOT / 'plantvillage'}",
        f"data.eval_target_dir={LOCAL_OVERLAP_ROOT / 'plantdoc'}",
    ], capture=True)
else:
    print("Set CONFIG['run_train'] = True to launch training from this notebook.")


In [ ]:
# ============================================================
# EVAL 1: PlantVillage -> PlantDoc overlap
# ============================================================

if CONFIG["run_eval_plantdoc"]:
    run([
        sys.executable, "-m", "phasephyto.evaluate",
        "--config", str(PLANTDOC_CFG),
        "--checkpoint", str(Path(CONFIG["checkpoint_dir"]) / "best_model.pt"),
        "--source-dir", str(LOCAL_OVERLAP_ROOT / "plantvillage"),
        "--target-dir", str(LOCAL_OVERLAP_ROOT / "plantdoc"),
        "--output", str(Path(CONFIG["checkpoint_dir"]) / "eval_plantdoc.json"),
    ], capture=True)
else:
    print("Set CONFIG['run_eval_plantdoc'] = True to run PlantDoc evaluation.")


In [ ]:
# ============================================================
# EVAL 2: PlantVillage -> Plant Pathology 2021 overlap
# ============================================================
# Note: this evaluates the SAME checkpoint (trained against PlantDoc) on the
# PP2021 target, using the PP2021 config only for class-name mapping. There is
# only one trained model in this pipeline.

if CONFIG["run_eval_pp2021"]:
    run([
        sys.executable, "-m", "phasephyto.evaluate",
        "--config", str(PP2021_CFG),
        "--checkpoint", str(Path(CONFIG["checkpoint_dir"]) / "best_model.pt"),
        "--source-dir", str(LOCAL_OVERLAP_ROOT / "plantvillage"),
        "--target-dir", str(LOCAL_OVERLAP_ROOT / "plant_pathology_2021"),
        "--output", str(Path(CONFIG["checkpoint_dir"]) / "eval_pp2021.json"),
    ], capture=True)
else:
    print("Set CONFIG['run_eval_pp2021'] = True to run Plant Pathology 2021 evaluation.")


## Combined evaluation summary

This cell reads `eval_plantdoc.json` and `eval_pp2021.json` if present, writes a combined JSON + CSV summary to Drive, and saves a simple comparison bar chart.


In [ ]:
# ============================================================
# COMBINED EVALUATION SUMMARY (saved to Drive)
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd

checkpoint_dir = Path(CONFIG["checkpoint_dir"])
plantdoc_eval = checkpoint_dir / "eval_plantdoc.json"
pp2021_eval = checkpoint_dir / "eval_pp2021.json"

rows = []
for target_name, eval_path in [
    ("plantdoc", plantdoc_eval),
    ("plant_pathology_2021", pp2021_eval),
]:
    if not eval_path.exists():
        print(f"Missing eval file for {target_name}: {eval_path}")
        continue
    payload = json.loads(eval_path.read_text())
    rows.append({
        "target": target_name,
        "source_accuracy": payload.get("source", {}).get("accuracy"),
        "target_accuracy": payload.get("target", {}).get("accuracy"),
        "accuracy_drop": payload.get("delta", {}).get("accuracy_drop"),
        "source_f1_macro": payload.get("source", {}).get("f1_macro"),
        "target_f1_macro": payload.get("target", {}).get("f1_macro"),
        "f1_drop": payload.get("delta", {}).get("f1_drop"),
        "use_case": payload.get("use_case"),
        "eval_json": str(eval_path),
    })

if not rows:
    raise RuntimeError(
        "No evaluation JSON files found. Run at least one eval cell first."
    )

summary_df = pd.DataFrame(rows).sort_values("target").reset_index(drop=True)
summary_json = checkpoint_dir / "apple_overlap_eval_summary.json"
summary_csv = checkpoint_dir / "apple_overlap_eval_summary.csv"
summary_md_path = checkpoint_dir / "apple_overlap_eval_summary.md"
summary_plot = checkpoint_dir / "apple_overlap_eval_summary.png"

summary_json.write_text(json.dumps(rows, indent=2) + "\n")
summary_df.to_csv(summary_csv, index=False)

bar_df = summary_df.set_index("target")[["target_accuracy", "target_f1_macro"]]
ax = bar_df.plot(kind="bar", figsize=(8, 4), ylim=(0, 1), rot=0)
ax.set_title("Apple overlap target performance")
ax.set_ylabel("score")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(summary_plot, dpi=140, bbox_inches="tight")
plt.show()

summary_lines = [
    "# Apple Overlap Evaluation Summary",
    "",
    f"- Checkpoint dir: `{checkpoint_dir}`",
    f"- PlantDoc eval: {'present' if plantdoc_eval.exists() else 'missing'} -- `{plantdoc_eval}`",
    f"- Plant Pathology 2021 eval: {'present' if pp2021_eval.exists() else 'missing'} -- `{pp2021_eval}`",
    "",
    "| Target | Target acc | Target F1 | Acc drop | F1 drop |",
    "|---|---:|---:|---:|---:|",
]
for row in rows:
    summary_lines.append(
        f"| {row['target']} | {row['target_accuracy']:.4f} | {row['target_f1_macro']:.4f} | {row['accuracy_drop']:+.4f} | {row['f1_drop']:+.4f} |"
    )
summary_md_path.write_text("\n".join(summary_lines) + "\n")

print("Saved combined overlap summary files:")
print("  JSON:", summary_json)
print("  CSV: ", summary_csv)
print("  MD:  ", summary_md_path)
print("  PNG: ", summary_plot)
summary_df


## Optional next step: batch inference notebook

After you have the four ablation checkpoints, open:

- `notebooks/PhasePhyto_Batch_Inference.ipynb`

and configure `DATASET_RUNS` with:

- `/content/data/overlap/apple_strict/plantdoc`
- `/content/data/overlap/apple_strict/plant_pathology_2021`

using `/content/data/overlap/apple_strict/plantvillage` as `class_to_idx_source`.
